In [ ]:
import time 
import warnings
import numpy as np 
import pandas as pd
import requests
import matplotlib.pyplot as plt

import torch
import torch.nn as nnfrom torch.utils.data import TensorDataset, DataLoader 
from sklearn.linear_model import Ridge

In [ ]:
import requests, pandas as pd, time

url = "https://api.binance.com/api/v3/klines"

all_rows = []
current = int(pd.to_datetime("2017-08-17", utc=True).timestamp() * 1000)
end_ms  = int(pd.to_datetime("2026-09-29", utc=True).timestamp() * 1000)

while current < end_ms:
    res = requests.get(url, params={
        "symbol": "BTCUSDT", "interval": "1h",
        "startTime": current, "endTime": end_ms, "limit": 1000
    }).json()

    all_rows.extend(res)
    current = res[-1][0] + 1
    time.sleep(0.03)
    print(f"fetched {len(all_rows)} rows")

print(f"\nTotal: {len(all_rows)} rows")
print(all_rows[0])

In [ ]:
columns = [
    "open_time", "open", "high", "low", "close", "volume",
    "close_time", "quote_asset_volume", "number_of_trades",
    "taker_buy_base_asset_volume", "taker_buy_quote_asset_volume", "ignore"
]
df = pd.DataFrame(all_rows, columns=columns)
print(df.head())

In [ ]:
df = df.drop(columns = ["ignore"])

In [ ]:
print(df.head())

In [ ]:
df["timetamp"] = pd.to_datetime(df["open_time"] , unit="ms" , utc=True )

for c in [
    "open_time", "open", "high", "low", "close", "volume",
    "close_time", "quote_asset_volume", "number_of_trades",
    "taker_buy_base_asset_volume", "taker_buy_quote_asset_volume"
]: 
    df[c] = df[c].astype(float)

In [ ]:
df = df.sort_values("timetamp").drop_duplicates("timetamp").reset_index(drop=True) 


In [ ]:
print(df.columns.tolist())

In [ ]:
print(df[["volume", "quote_asset_volume"]].describe())
print(f"Mean price implied: {(df['quote_asset_volume'] / df['volume']).mean():.0f}")

print(df["number_of_trades"].describe())

violations = (df["taker_buy_base_asset_volume"] > df["volume"] * 1.001).sum()
print(f"Violations: {violations}")   

df["taker_buy_ratio"] = (
    df["taker_buy_base_asset_volume"] / (df["volume"] + 1e-8)
)
print(df["taker_buy_ratio"].describe())

In [ ]:
df["return"] = ((df["close"]-df["open"] )/df["open"] )*100

In [ ]:
df["hourly_range"] = (df["high"] - df["low"]) / df["low"] * 100

In [ ]:
df["rt_cc"] = ((df["close"] - df["close"].shift(1)) / df["close"].shift(1)) * 100

In [ ]:
df["range_ma_24"] = df["hourly_range"].rolling(24).mean()df["range_ma_168"] = df["hourly_range"].rolling(168).mean()

In [ ]:
df["volatility_24"] = df["rt_cc"].rolling(24).std()

In [ ]:
df["volume_ma_24"] = df["volume"].rolling(24).mean()
df["volume_ratio"] = df["volume"] / (df["volume_ma_24"] + 1e-8)

In [ ]:
tau = 2 * np.pi
hour = df["timetamp"].dt.hour
dow = df["timetamp"].dt.dayofweek

df[["hour_sin", "hour_cos"]] = np.column_stack(
    [np.sin(tau * hour / 24), np.cos(tau * hour / 24)]
)

df[["dow_sin", "dow_cos"]] = np.column_stack(
    [np.sin(tau * dow / 7), np.cos(tau * dow / 7)]
)

In [ ]:
df["volume_change"] = df["volume"].pct_change() * 100

df["range_ma_720"] = df["hourly_range"].rolling(720).mean()
df["volatility_168"] = df["rt_cc"].rolling(168).std()

df["ma_24_price"]  = df["close"].rolling(24).mean()
df["ma_168_price"] = df["close"].rolling(168).mean()
df["distance_ma_24"]  = (df["close"] - df["ma_24_price"])  / (df["ma_24_price"]  + 1e-8)
df["distance_ma_168"] = (df["close"] - df["ma_168_price"]) / (df["ma_168_price"] + 1e-8)

for lag in (1, 2, 3, 6, 12):
    df[f"range_lag_{lag}"] = df["hourly_range"].shift(lag)

df["is_weekend"] = (df["timetamp"].dt.dayofweek >= 5).astype(float)

In [ ]:
df["taker_buy_ratio"] = (
    df["taker_buy_base_asset_volume"] / (df["volume"] + 1e-8)
)
df["order_flow_imbalance"] = (
    (2 * df["taker_buy_base_asset_volume"] - df["volume"])
    / (df["volume"] + 1e-8)
)
df["avg_trade_size"]     = df["volume"] / (df["number_of_trades"] + 1e-8)
df["avg_trade_size_usd"] = df["quote_asset_volume"] / (df["number_of_trades"] + 1e-8)

In [ ]:
df["vol_regime"]      = df["volatility_24"] / (df["volatility_168"] + 1e-8)
df["range_vs_ma_168"] = df["hourly_range"] / (df["range_ma_168"] + 1e-8)
df["range_vs_ma_720"] = df["hourly_range"] / (df["range_ma_720"] + 1e-8)

df["high_vol_streak"] = (
    (df["hourly_range"] > df["range_ma_24"]).rolling(6).sum()
)

df["roll_max_720"]  = df["close"].rolling(720).max()
df["drawdown_720"]  = (df["close"] - df["roll_max_720"]) / (df["roll_max_720"] + 1e-8)

df["return_skew_168"] = df["rt_cc"].rolling(168).skew()

In [ ]:
df["trade_count_change"] = df["number_of_trades"].pct_change() * 100

In [ ]:
df["vwap_hour"]     = df["quote_asset_volume"] / (df["volume"] + 1e-8)
df["close_vs_vwap"] = (df["close"] - df["vwap_hour"]) / (df["vwap_hour"] + 1e-8) * 100

In [ ]:
df["ofi_ma_24"]        = df["order_flow_imbalance"].rolling(24).mean()
df["ofi_std_24"]       = df["order_flow_imbalance"].rolling(24).std()
df["taker_buy_ma_24"]  = df["taker_buy_ratio"].rolling(24).mean()
df["trade_size_ma_24"] = df["avg_trade_size"].rolling(24).mean()
df["trade_size_ratio"] = df["avg_trade_size"] / (df["trade_size_ma_24"] + 1e-8)

In [ ]:
for lag in (1, 2, 3):
    df[f"trade_ratio_lag_{lag}"] = df["taker_buy_ratio"].shift(lag)

In [ ]:
for lag in (1, 2, 3, 6, 12):
    df[f"ofi_lag_{lag}"] = df["order_flow_imbalance"].shift(lag)

In [ ]:
df["next_hour_range"] = df["hourly_range"].shift(-1)

In [ ]:
df["target"] = df["next_hour_range"] / (df["range_ma_24"] + 1e-8)
df["target"] = df["target"].clip(0, 100)

In [ ]:
def fetch_funding(start_ms, end_ms):
    records, cur_time = [], start_ms
    while cur_time < end_ms:
        res = requests.get("https://fapi.binance.com/fapi/v1/fundingRate",
                           params={"symbol": "BTCUSDT",
                                   "startTime": cur_time, "limit": 1000},
                           timeout=10)
        data = res.json()
        if not data:
            break
        records.extend(data)
        cur_time = data[-1]["fundingTime"] + 1
        time.sleep(0.02)

    df_f = pd.DataFrame(records)
    df_f["timestamp"] = pd.to_datetime(df_f["fundingTime"], unit="ms", utc=True)
    df_f["funding_rate"] = df_f["fundingRate"].astype(float)
    df_f["funding_rate_change"] = df_f["funding_rate"].diff()
    mean_8h = df_f["funding_rate"].rolling(21, min_periods=7).mean()
    std_8h  = df_f["funding_rate"].rolling(21, min_periods=7).std() + 1e-8
    df_f["z_funding"] = (df_f["funding_rate"] - mean_8h) / std_8h
    return df_f[["timestamp", "funding_rate", "funding_rate_change", "z_funding"]]

start_ms = int(df["timetamp"].min().timestamp() * 1000)
end_ms   = int(df["timetamp"].max().timestamp() * 1000)
df_funding = fetch_funding(start_ms, end_ms)

df = pd.merge_asof(df.sort_values("timetamp"),
                   df_funding.sort_values("timestamp"),
                   left_on="timetamp", right_on="timestamp",
                   direction="backward")

df[["funding_rate", "funding_rate_change", "z_funding"]] = (
    df[["funding_rate", "funding_rate_change", "z_funding"]].ffill().fillna(0)
)


In [ ]:
cols_ohlcv = [
    "return",              
    "hourly_range",       
    "rt_cc",            
    "volume_change",       
    "volume_ratio",    
    "range_ma_24",
    "range_ma_168",
    "range_ma_720",
    "volatility_24",
    "volatility_168",
    "distance_ma_24",
    "distance_ma_168",
    "hour_sin", "hour_cos",
    "dow_sin",  "dow_cos",
    "is_weekend",
    "range_lag_1", "range_lag_2", "range_lag_3",
    "range_lag_6", "range_lag_12",
    "vol_regime",
    "range_vs_ma_168",
    "range_vs_ma_720",
    "high_vol_streak",
    "drawdown_720",
    "return_skew_168",
]
cols_orderflow = [
    "taker_buy_ratio",         
    "order_flow_imbalance",   
    "close_vs_vwap",          
    "avg_trade_size",         
    "avg_trade_size_usd",     
    "trade_size_ratio",       
    "trade_count_change", 
    "ofi_ma_24",
    "ofi_std_24",
    "taker_buy_ma_24",
    "trade_size_ma_24",
    "ofi_lag_1", "ofi_lag_2", "ofi_lag_3", "ofi_lag_6", "ofi_lag_12",
    "trade_ratio_lag_1", "trade_ratio_lag_2", "trade_ratio_lag_3",
]

cols_funding = ["funding_rate", "funding_rate_change", "z_funding"]

feature_cols = cols_ohlcv + cols_orderflow + cols_funding

In [ ]:
needed = feature_cols + ["target"]

df_clean = df.replace([np.inf, -np.inf], np.nan)

df_clean = df_clean.dropna(subset=needed).reset_index(drop=True)

print("Total rows:", len(df_clean))
print("Features:", len(feature_cols))
print("First date:", df_clean["timetamp"].min())
print("Last date: ", df_clean["timetamp"].max())

In [ ]:
split_point = int(len(df_clean) * 0.8)

x = df_clean[feature_cols].values.astype("float32")
y = df_clean["target"].values.astype("float32")  

x_train = x[:split_point]
x_test  = x[split_point:]
y_train = y[:split_point]
y_test  = y[split_point:]

In [ ]:
mean = x_train.mean(axis=0)

std = x_train.std(axis=0) + 1e-8

x_train = (x_train - mean) / std

x_test = (x_test - mean) / std

y_train_raw = y_train.copy()
y_test_raw = y_test.copy()

print("X_train shape:", x_train.shape)
print("X_test shape: ", x_test.shape)


In [ ]:
model = nn.Sequential(
    nn.Linear(len(feature_cols), 32),
    nn.ReLU(),
    nn.Linear(32, 1),
)

print(model)

In [ ]:
x_t = torch.tensor(x_train, dtype=torch.float32)
y_t = torch.tensor(y_train, dtype=torch.float32).reshape(-1, 1)

loader = DataLoader(TensorDataset(x_t, y_t), batch_size=128, shuffle=True)

loss_fn = nn.MSELoss()
optimizer = torch.optim.Adam(model.parameters(), lr=0.001)


In [ ]:
for epoch in range(30):
    total_loss = 0.0

    for xb, yb in loader:
        optimizer.zero_grad()       
        preds = model(xb)            
        loss = loss_fn(preds, yb)    
        loss.backward()             
        optimizer.step()       

        total_loss += loss.item()

    avg = total_loss / len(loader)
    print("epoch", epoch + 1, "| loss", round(avg, 5))

In [ ]:
with torch.no_grad():
    x_test_t = torch.tensor(x_test)
    preds_log = model(x_test_t).numpy().flatten()

preds = preds_log

def r2(real, guess):
    top    = ((real - guess) ** 2).sum()
    bottom = ((real - real.mean()) ** 2).sum()
    return 1 - top / bottom

nn_r2    = r2(y_test, preds)
naive    = np.full(len(y_test), y_train.mean())
naive_r2 = r2(y_test, naive)

print("NN R²:    ", round(nn_r2, 4))
print("Naive R²: ", round(naive_r2, 4))

print("Corr:", np.corrcoef(y_test, preds)[0, 1])

In [ ]:
n = 300 

plt.figure(figsize=(12, 4))
plt.plot(y_test[:n], label="Actual", alpha=0.7)
plt.plot(preds[:n], label="NN prediction", alpha=0.8)
plt.title("BTC Hourly Volatility Ratio — Test Set")
plt.xlabel("Hour")
plt.ylabel("Ratio")
plt.legend()
plt.tight_layout()
plt.show()

In [ ]:
split_point = int(len(df_clean) * 0.8)

x = df_clean[feature_cols].values.astype("float32")
y = df_clean["target"].values.astype("float32")

x_train = x[:split_point]
x_test = x[split_point:]

y_train = y[:split_point]
y_test = y[split_point:]

import xgboost as xgb
from sklearn.model_selection import train_test_split
from sklearn.metrics import accuracy_score, classification_report
from sklearn.metrics import mean_squared_error, mean_absolute_error, r2_score

model = xgb.XGBRegressor(
     n_estimators=100,    
    max_depth=5,           
    learning_rate=0.1,    
    objective='reg:squarederror',
    random_state=42,
)

model.fit(
    x_train,y_train,
    eval_set=[(x_test, y_test)],
    verbose = False
)

xgb_predict = model.predict(x_test)

mse = mean_squared_error(y_test, xgb_predict)
mae = mean_absolute_error(y_test, xgb_predict)
r2 = r2_score(y_test, xgb_predict)

print(f"Mean Absolute Error (MAE): {mae:.4f}")
print(f"Mean Squared Error (MSE): {mse:.4f}")
print(f"R² Score (Variance Explained): {r2:.4f}")


In [ ]:
import numpy as np
from sklearn.linear_model import Ridge
from sklearn.metrics import mean_squared_error, mean_absolute_error, r2_score
from sklearn.preprocessing import StandardScaler

split_point = int(len(df_clean) * 0.8)

x = df_clean[feature_cols].values.astype("float32")
y = df_clean["target"].values.astype("float32")

x_train = x[:split_point]
x_test = x[split_point:]

y_train = y[:split_point]
y_test = y[split_point:]

scaler = StandardScaler()
x_train_scaled = scaler.fit_transform(x_train)
x_test_scaled = scaler.transform(x_test)
ridge_model  = Ridge(alpha=1.0)
ridge_model.fit(x_train_scaled, y_train)

y_pred = ridge_model.predict(x_test_scaled)
mse = mean_squared_error(y_test, y_pred)

print("Model Coefficients:", ridge_model.coef_)
print("Mean Squared Error:", mse)

ridge_pred = ridge_model.predict(x_test_scaled)

print("Ridge MAE:", mean_absolute_error(y_test, ridge_pred))
print("Ridge MSE:", mean_squared_error(y_test, ridge_pred))
print("Ridge R²:", r2_score(y_test, ridge_pred))


In [ ]:
n = 300 

plt.figure(figsize=(12, 4))
plt.plot(y_test[:n], label="Actual", alpha=0.7)
plt.plot(preds[:n], label="NN prediction", alpha=0.8)
plt.plot(ridge_pred[:n], label = "Ridge prediction" , alpha=0.8)
plt.plot(xgb_predict[:n],label = "XGboost prediction",alpha = 0.8)
plt.title("BTC Hourly Volatility Ratio — Test Set")
plt.xlabel("Hour")
plt.ylabel("Ratio")
plt.legend()
plt.tight_layout()
plt.show()

In [ ]:
n = 300

actual = np.asarray(y_test[:n]).ravel()
nn = np.asarray(preds[:n]).ravel()
ridge = np.asarray(ridge_pred[:n]).ravel()
xgb = np.asarray(xgb_predict[:n]).ravel()

all_values = np.concatenate([actual, nn, ridge, xgb])
min_value = all_values.min()
max_value = all_values.max()

fig, axes = plt.subplots(1, 3, figsize=(15, 4))
models = [
    ("Neural Network", nn),
    ("Ridge", ridge),
    ("XGBoost", xgb)
]

for ax, (name, prediction) in zip(axes, models):
    ax.scatter(actual, prediction, alpha=0.5, s=25)
    ax.plot([min_value, max_value], [min_value, max_value], linestyle="--")
    ax.set_title(name)
    ax.set_xlabel("Actual")
    ax.set_ylabel("Predicted")
    ax.grid(alpha=0.3)

fig.suptitle("BTC Hourly Volatility Ratio — Actual vs Predicted", fontsize=14)
plt.tight_layout()
plt.show()
